In [ ]:
import pandas as pd
import numpy as np
from jamo import h2j, j2hcj
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm
import gc

# ==========================================
# 1. 자모 분리 함수 (공백 유지)
# ==========================================
def decompose_text(text):
    # 한글 -> 자모 분리 (jamo 라이브러리)
    # 띄어쓰기는 유지해야 단어 구분이 됨
    return j2hcj(h2j(text))

# ==========================================
# 2. 메인 파이프라인 (검색 기반 교정)
# ==========================================
def run_pipeline():
    try:
        train_df = pd.read_csv('train.csv')
        test_df = pd.read_csv('test.csv')
        print("✅ 데이터 로드 완료")
    except FileNotFoundError:
        print("❌ csv 파일이 없습니다.")
        return

    print("\n--- [Step 1] 정답 단어장(DB) 구축 ---")
    # 정답 데이터(output)에 있는 모든 '문장' 혹은 '단어'를 DB로 만듭니다.
    # 문장 단위로 매칭하면 정확도가 높지만, 본 적 없는 문장은 못 맞춥니다.
    # 단어 단위로 쪼개서 매칭하는 것이 일반화 성능이 좋습니다.
    
    # 1. train_df의 output(정답)에서 유니크한 단어 수집
    correct_sentences = train_df['output'].astype(str).tolist()
    
    # 단어 단위로 쪼개서 DB 구축 (중복 제거)
    # 예: "안녕하세요 반갑습니다" -> ["안녕하세요", "반갑습니다"]
    correct_words = set()
    for sent in correct_sentences:
        for word in sent.split():
            correct_words.add(word)
            
    # set을 리스트로 변환 (검색 대상 DB)
    db_words = sorted(list(correct_words))
    print(f"📚 정답 단어장 크기: {len(db_words):,}개")

    # DB의 단어들을 자모로 분해하여 벡터화 준비
    db_words_jamo = [decompose_text(w) for w in db_words]

    print("\n--- [Step 2] TF-IDF 벡터화 (자모 N-gram) ---")
    # analyzer='char': 글자 단위 분석
    # ngram_range=(1, 3): 1글자~3글자 단위로 쪼개서 유사도 비교 (자모 패턴 인식)
    # 예: '안녕' -> 'ㅇ', 'ㅏ', 'ㄴ', 'ㅇㅏ', 'ㅏㄴ', ...
    vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), min_df=1)
    
    # DB 벡터화 (학습)
    db_vectors = vectorizer.fit_transform(db_words_jamo)
    
    print("\n--- [Step 3] 검색 엔진 생성 (k-NN) ---")
    # 유클리드 거리보다 코사인 유사도가 텍스트 매칭에 유리하지만,
    # sklearn의 NearestNeighbors는 기본적으로 거리 기반입니다.
    # metric='cosine'을 쓰거나, brute force 알고리즘 사용
    nbrs = NearestNeighbors(n_neighbors=1, algorithm='brute', metric='cosine', n_jobs=-1)
    nbrs.fit(db_vectors)
    
    print("\n--- [Step 4] 테스트 데이터 교정 (검색) ---")
    predictions = []
    
    # 배치 사이즈 설정 (한 번에 처리할 단어 수, 메모리 고려)
    BATCH_SIZE = 1000 
    
    # 테스트 데이터를 문장 단위로 순회
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Correction"):
        input_sentence = str(row['input'])
        input_words = input_sentence.split() # 띄어쓰기 기준으로 단어 분리
        
        restored_words = []
        
        # 문장 내의 각 단어(오타)에 대해 가장 비슷한 정답 단어 검색
        if not input_words:
            predictions.append("")
            continue
            
        # 1. 단어들을 자모로 분해
        input_words_jamo = [decompose_text(w) for w in input_words]
        
        # 2. 벡터화 (transform) - DB 만들 때 쓴 vectorizer 그대로 사용
        try:
            input_vectors = vectorizer.transform(input_words_jamo)
            
            # 3. 가장 가까운 단어 검색
            # distances: 거리 (0에 가까울수록 비슷), indices: DB 내 인덱스
            distances, indices = nbrs.kneighbors(input_vectors)
            
            for i, word_idx in enumerate(indices):
                dist = distances[i][0]
                best_match_idx = word_idx[0]
                best_word = db_words[best_match_idx]
                
                # 유사도 임계값 설정 (옵션)
                # 거리가 너무 멀면(완전 딴소리면) 원본 유지 or 그래도 가장 가까운 것 사용
                # Cosine distance는 0~1 사이 (0이 완전 일치)
                if dist < 0.6: # 꽤 비슷하면 교정
                    restored_words.append(best_word)
                else: # 너무 다르면 원본 유지 (안전장치)
                    restored_words.append(input_words[i])
                    
        except Exception as e:
            # 에러 시 원본 단어 사용
            restored_words = input_words
            
        # 다시 문장으로 조립
        restored_sentence = " ".join(restored_words)
        predictions.append(restored_sentence)

    submission = pd.DataFrame({'ID': test_df['ID'], 'output': predictions})
    submission.to_csv('submission_knn.csv', index=False, encoding='utf-8-sig')
    print("\n✅ 완료! 검색 기반 교정 파일 'submission_knn.csv' 생성됨.")

if __name__ == "__main__":
    run_pipeline()